<!-- PROFESSIONAL_HEADER_v1 -->
<div align="center">

# 🚛 FleetLogix
## Avance 3 · Data Warehouse
### *Modelo dimensional Snowflake (star schema) y carga ETL*

**Henry Data Science · Proyecto 2 · Dody Dueñas**  
📧 dodydurema67@gmail.com · 🗓 2026

</div>

---

## 🎯 Objetivos de aprendizaje

- Diseñar un modelo dimensional star schema sobre Snowflake
- Definir tablas de hechos (fact_trips, fact_deliveries) y dimensiones
- Implementar la carga incremental desde PostgreSQL
- Documentar SCD tipo 1 y 2 donde aplique

## 📦 Entregables

- `sql/snowflake_schema.sql` con DDL de DW
- `scripts/05_etl_pipeline.py` con la carga ETL
- Diagrama del modelo estrella

## 🧭 Cómo navegar este notebook

1. Ejecuta las celdas de **arriba hacia abajo** la primera vez.
2. Cada bloque tiene una celda markdown que explica el *por qué* antes del código.
3. Los outputs incluyen métricas, tiempos y validaciones.
4. Si interrumpes, todas las operaciones de BD usan transactions y son seguras de reintentar.

## 🔗 Pre-requisitos

- Haber corrido **`Avance_0_QuickStart.ipynb`** y verificado que todos los chequeos están en verde.
- PostgreSQL corriendo via `docker compose up -d` o el script `dashboard/setup/setup.ps1`.

---


<div class="header">
<h1>❄️ FleetLogix Enterprise - Data Warehouse y ETL</h1>
<h2>Avance 3 - Proyecto Integrador M2 - Henry Data Science</h2>
<div style="display:flex; justify-content:space-between; margin-top:20px;">
<span>👤 <strong>Autor:</strong> Dody Dueñas</span>
<span>📅 <strong>Fecha:</strong> Abril 2026</span>
<span>🎓 <strong>Campus:</strong> Henry Data Science</span>
</div>
</div>

<div class="section">
<h2>📋 1. Resumen Ejecutivo</h2>
<p>Este notebook documenta la fase de <strong>Modelado Dimensional y Pipeline ETL</strong> para el ecosistema FleetLogix.</p>
<h3>🎯 Objetivos del Avance</h3>
<ul>
<li><span class="badge badge-success">✅</span> Diseño e implementación de modelo en estrella (Star Schema)</li>
<li><span class="badge badge-success">✅</span> Creación de esquema en Snowflake</li>
<li><span class="badge badge-success">✅</span> Pipeline ETL automatizado (Extract, Transform, Load)</li>
<li><span class="badge badge-success">✅</span> Integración de telemetría IoT simulada en MongoDB</li>
<li><span class="badge badge-success">✅</span> Control de calidad de datos en el pipeline</li>
</ul>
<table>
<tr><th>Componente</th><th>Descripción</th></tr>
<tr><td>Star Schema</td><td>4 dimensiones + 2 hechos</td></tr>
<tr><td>Snowflake</td><td>FLEETLOGIX_DW.ANALYTICS</td></tr>
<tr><td>ETL Pipeline</td><td>Extract → Transform → Load</td></tr>
<tr><td>IoT</td><td>6,500+ eventos simulados</td></tr>
</table>
</div>

<div class="section">
<h2>🏗️ 2. Arquitectura del Data Warehouse - Star Schema</h2>

<div align="center" style="background-color: #121212; padding: 30px; border-radius: 12px; border: 1px solid #333; margin: 25px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.5);">
    <h2 style="color: #38ef7d; font-family: 'Segoe UI', sans-serif; text-transform: uppercase; letter-spacing: 2px;">🏗️ Arquitectura de Data Warehouse (Star Schema)</h2>
    <p style="color: #888; font-size: 14px; margin-bottom: 25px;">Esquema multidimensional optimizado en Snowflake para consultas de baja latencia.</p>
    <img src="../docs/assets/snowflake_star_schema_1776287115646.png" style="max-width: 90%; border-radius: 8px; box-shadow: 0 5px 15px rgba(0,0,0,0.8);" />
</div>

### Tablas de Dimensiones:
| Tabla | Descripción | Atributos Clave |
|-------|-------------|-----------------|
| **DIM_VEHICLE** | Dimensión de vehículos (SCD2) | `vehicle_key, license_plate, vehicle_type` |
| **DIM_DRIVER** | Dimensión de conductores (SCD2) | `driver_key, employee_code, first_name` |
| **DIM_ROUTE** | Dimensión de rutas geo (SCD1) | `route_key, route_code, origin_city` |
| **DIM_DATE** | Dimensión temporal (SCD0) | `date_key, year, month, day, quarter` |

### Tabla de Hechos:
| Tabla | Descripción | Métricas Escalables |
|-------|-------------|---------------------|
| **FACT_DELIVERIES** | Hechos de entregas logísticas | `delivery_time_minutes, fuel_consumed_liters, delay_minutes` |


<h3>Tablas de Dimensiones:</h3>
<table>
<tr><th>Tabla</th><th>Descripción</th><th>Atributos Clave</th></tr>
<tr><td>DIM_VEHICLE</td><td>Dimensión de vehículos</td><td>vehicle_key, license_plate, vehicle_type</td></tr>
<tr><td>DIM_DRIVER</td><td>Dimensión de conductores</td><td>driver_key, employee_code, first_name</td></tr>
<tr><td>DIM_ROUTE</td><td>Dimensión de rutas</td><td>route_key, route_code, origin_city</td></tr>
<tr><td>DIM_DATE</td><td>Dimensión de tiempo</td><td>date_key, year, month, day, quarter</td></tr>
</table>

<h3>Tablas de Hechos:</h3>
<table>
<tr><th>Tabla</th><th>Descripción</th><th>Métricas</th></tr>
<tr><td>FACT_TRIPS</td><td>Hechos de viajes</td><td>fuel_consumed, total_weight, distance</td></tr>
<tr><td>FACT_DELIVERIES</td><td>Hechos de entregas</td><td>packages_count, delivered_weight, delivery_time</td></tr>
</table>
</div>

<div class="section">
<h2>⚙️ 3. Configuración del Entorno</h2>
</div>

In [ ]:
import os
import pandas as pd
import numpy as np
import psycopg2
import seaborn as sns
import matplotlib.pyplot as plt
import json
from datetime import datetime, timedelta
from faker import Faker

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

    PG_CONFIG = {'dbname': os.getenv('DB_NAME', 'fleetlogix_db'), 'user': os.getenv('DB_USER', 'postgres'), 'password': os.getenv('DB_PASSWORD', ''), 'host': os.getenv('DB_HOST', 'localhost'), 'port': int(os.getenv('DB_PORT', 5432))}
    SNOWFLAKE_CONFIG = {'user': os.getenv('SNOWFLAKE_USER'), 'password': os.getenv('SNOWFLAKE_PASSWORD'), 'account': os.getenv('SNOWFLAKE_ACCOUNT'), 'warehouse': os.getenv('SNOWFLAKE_WAREHOUSE', 'COMPUTE_WH'), 'database': os.getenv('SNOWFLAKE_DATABASE', 'FLEETLOGIX_DW'), 'schema': os.getenv('SNOWFLAKE_SCHEMA', 'ANALYTICS')}

print("✅ Librerías importadas")
print("✅ Configuración de conexiones cargada")

<div class="section">
<h2>📦 4. Extracción de Datos desde PostgreSQL</h2>
</div>

In [ ]:
pg_conn = psycopg2.connect(**PG_CONFIG)
print("✅ Conexión a PostgreSQL establecida")

print("\n📦 Extrayendo datos de PostgreSQL...")

df_vehicles = pd.read_sql("SELECT * FROM vehicles", pg_conn)
df_drivers = pd.read_sql("SELECT * FROM drivers", pg_conn)
df_routes = pd.read_sql("SELECT * FROM routes", pg_conn)
df_trips = pd.read_sql("SELECT * FROM trips LIMIT 10000", pg_conn)
df_deliveries = pd.read_sql("SELECT * FROM deliveries LIMIT 10000", pg_conn)
df_maintenance = pd.read_sql("SELECT * FROM maintenance", pg_conn)

print(f"   ✅ Vehicles: {len(df_vehicles):,}")
print(f"   ✅ Drivers: {len(df_drivers):,}")
print(f"   ✅ Routes: {len(df_routes):,}")
print(f"   ✅ Trips: {len(df_trips):,}")
print(f"   ✅ Deliveries: {len(df_deliveries):,}")
print(f"   ✅ Maintenance: {len(df_maintenance):,}")

<div class="section">
<h2>🔄 5. Transformación de Datos (ETL - Transform)</h2>
</div>

In [ ]:
print("=" * 70)
print("🔄 FASE DE TRANSFORMACIÓN - ETL")
print("=" * 70)

# DIM_VEHICLE
dim_vehicle = df_vehicles.copy()
dim_vehicle['vehicle_key'] = range(1, len(dim_vehicle) + 1)
dim_vehicle['load_timestamp'] = datetime.now()
print(f"✅ DIM_VEHICLE: {len(dim_vehicle)} registros transformados")

# DIM_DRIVER
dim_driver = df_drivers.copy()
dim_driver['driver_key'] = range(1, len(dim_driver) + 1)
dim_driver['load_timestamp'] = datetime.now()
print(f"✅ DIM_DRIVER: {len(dim_driver)} registros transformados")

# DIM_ROUTE
dim_route = df_routes.copy()
dim_route['route_key'] = range(1, len(dim_route) + 1)
dim_route['load_timestamp'] = datetime.now()
print(f"✅ DIM_ROUTE: {len(dim_route)} registros transformados")

# DIM_DATE
date_range = pd.date_range(start='2025-01-01', end='2025-12-31', freq='D')
dim_date = pd.DataFrame({'full_date': date_range})
dim_date['date_key'] = range(1, len(dim_date) + 1)
dim_date['year'] = dim_date['full_date'].dt.year
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['day'] = dim_date['full_date'].dt.day
dim_date['quarter'] = dim_date['full_date'].dt.quarter
dim_date['load_timestamp'] = datetime.now()
print(f"✅ DIM_DATE: {len(dim_date)} registros transformados")

# FACT_TRIPS
fact_trips = df_trips.copy()
fact_trips['fact_trip_key'] = range(1, len(fact_trips) + 1)
fact_trips['load_timestamp'] = datetime.now()
fact_trips['duration_hours'] = (pd.to_datetime(fact_trips['arrival_datetime']) - pd.to_datetime(fact_trips['departure_datetime'])).dt.total_seconds() / 3600
trips_with_routes = fact_trips.merge(df_routes[['route_id', 'distance_km']], on='route_id', how='left')
trips_with_routes['fuel_efficiency'] = (trips_with_routes['fuel_consumed_liters'] / trips_with_routes['distance_km'] * 100).round(2)
print(f"✅ FACT_TRIPS: {len(fact_trips)} registros transformados")

# FACT_DELIVERIES
fact_deliveries = df_deliveries.copy()
fact_deliveries['fact_delivery_key'] = range(1, len(fact_deliveries) + 1)
fact_deliveries['load_timestamp'] = datetime.now()
fact_deliveries['delivery_time_hours'] = (pd.to_datetime(fact_deliveries['delivered_datetime']) - pd.to_datetime(fact_deliveries['scheduled_datetime'])).dt.total_seconds() / 3600
print(f"✅ FACT_DELIVERIES: {len(fact_deliveries)} registros transformados")

<div class="section">
<h2>🔍 6. Control de Calidad (Data Quality Checks)</h2>
</div>

In [ ]:
print("=" * 70)
print("🔍 VERIFICACIONES DE CALIDAD DE DATOS")
print("=" * 70)

quality_checks = []

null_vehicle_keys = dim_vehicle['vehicle_key'].isnull().sum()
quality_checks.append(("DIM_VEHICLE - Keys nulos", null_vehicle_keys, null_vehicle_keys == 0))
null_driver_keys = dim_driver['driver_key'].isnull().sum()
quality_checks.append(("DIM_DRIVER - Keys nulos", null_driver_keys, null_driver_keys == 0))
dup_vehicle_keys = dim_vehicle['vehicle_key'].duplicated().sum()
quality_checks.append(("DIM_VEHICLE - Keys duplicados", dup_vehicle_keys, dup_vehicle_keys == 0))
negative_fuel = (fact_trips['fuel_consumed_liters'] < 0).sum()
quality_checks.append(("FACT_TRIPS - Consumo negativo", negative_fuel, negative_fuel == 0))
negative_weight = (fact_trips['total_weight_kg'] < 0).sum()
quality_checks.append(("FACT_TRIPS - Peso negativo", negative_weight, negative_weight == 0))
negative_delivery_time = (fact_deliveries['delivery_time_hours'] < 0).sum()
quality_checks.append(("FACT_DELIVERIES - Tiempo negativo", negative_delivery_time, negative_delivery_time == 0))

print(f"\n{'Verificación':<45} {'Encontrados':>12} {'Estado':>12}")
print("-" * 70)
all_passed = True
for check_name, count, passed in quality_checks:
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"{check_name:<45} {count:>12} {status:>12}")
    if not passed: all_passed = False
print("-" * 70)
print(f"\nResultado Final: {'✅ TODAS LAS VALIDACIONES PASS' if all_passed else '❌ HAY FALLOS'}")

<div class="section">
<h2>❄️ 7. Esquema SQL para Snowflake</h2>
</div>

In [ ]:
print("=" * 70)
print("❄️  ESQUEMA STAR SCHEMA PARA SNOWFLAKE")
print("=" * 70)

snowflake_sql = """
-- =====================================================
-- FLEETLOGIX DWH - SNOWFLAKE STAR SCHEMA
-- Autor: Dody Dueñas - Proyecto Integrador M2
-- =====================================================

-- CREACIÓN DE DATABASE Y SCHEMA
CREATE DATABASE IF NOT EXISTS FLEETLOGIX_DW;
USE DATABASE FLEETLOGIX_DW;
CREATE SCHEMA IF NOT EXISTS ANALYTICS;
USE SCHEMA ANALYTICS;

-- DIM_VEHICLE
CREATE OR REPLACE TABLE DIM_VEHICLE (
    vehicle_key INTEGER PRIMARY KEY,
    vehicle_id INTEGER,
    license_plate VARCHAR(20),
    vehicle_type VARCHAR(50),
    capacity_kg DECIMAL(10,2),
    fuel_type VARCHAR(20),
    acquisition_date DATE,
    status VARCHAR(20),
    load_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

-- DIM_DRIVER
CREATE OR REPLACE TABLE DIM_DRIVER (
    driver_key INTEGER PRIMARY KEY,
    driver_id INTEGER,
    employee_code VARCHAR(20),
    first_name VARCHAR(100),
    last_name VARCHAR(100),
    license_number VARCHAR(50),
    license_expiry DATE,
    phone VARCHAR(20),
    hire_date DATE,
    status VARCHAR(20),
    load_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

-- DIM_ROUTE
CREATE OR REPLACE TABLE DIM_ROUTE (
    route_key INTEGER PRIMARY KEY,
    route_id INTEGER,
    route_code VARCHAR(20),
    origin_city VARCHAR(100),
    destination_city VARCHAR(100),
    distance_km DECIMAL(10,2),
    estimated_duration_hours DECIMAL(5,2),
    toll_cost DECIMAL(10,2),
    load_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

-- DIM_DATE
CREATE OR REPLACE TABLE DIM_DATE (
    date_key INTEGER PRIMARY KEY,
    full_date DATE,
    year INTEGER,
    month INTEGER,
    day INTEGER,
    quarter INTEGER,
    load_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

-- FACT_TRIPS
CREATE OR REPLACE TABLE FACT_TRIPS (
    fact_trip_key INTEGER PRIMARY KEY,
    trip_id INTEGER,
    vehicle_key INTEGER,
    driver_key INTEGER,
    route_key INTEGER,
    departure_datetime TIMESTAMP,
    arrival_datetime TIMESTAMP,
    fuel_consumed_liters DECIMAL(10,2),
    total_weight_kg DECIMAL(10,2),
    status VARCHAR(20),
    distance_km DECIMAL(10,2),
    duration_hours DECIMAL(10,2),
    fuel_efficiency DECIMAL(10,2),
    load_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

-- FACT_DELIVERIES
CREATE OR REPLACE TABLE FACT_DELIVERIES (
    fact_delivery_key INTEGER PRIMARY KEY,
    delivery_id INTEGER,
    trip_id INTEGER,
    tracking_number VARCHAR(50),
    customer_name VARCHAR(200),
    delivery_address TEXT,
    package_weight_kg DECIMAL(10,2),
    scheduled_datetime TIMESTAMP,
    delivered_datetime TIMESTAMP,
    delivery_status VARCHAR(20),
    recipient_signature BOOLEAN,
    delivery_time_hours DECIMAL(10,2),
    load_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

-- VISTAS ANALÍTICAS
CREATE OR REPLACE VIEW V_RENDIMIENTO_VEHICULOS AS
SELECT v.vehicle_type, v.license_plate, COUNT(t.trip_id) AS total_viajes,
       SUM(t.fuel_consumed_liters) AS consumo_total, ROUND(AVG(t.fuel_efficiency), 2) AS eficiencia_promedio
FROM DIM_VEHICLE v LEFT JOIN FACT_TRIPS t ON v.vehicle_key = t.vehicle_key
GROUP BY v.vehicle_type, v.license_plate;
"""

print("✅ SQL para Snowflake generado")
print("\nEste SQL debe ejecutarse en Snowflake para crear el Star Schema.")

<div class="section">
<h2>📡 8. Integración de Telemetría IoT (MongoDB)</h2>
</div>

In [ ]:
print("=" * 70)
print("📡 GENERACIÓN DE TELEMETRÍA IoT (Simulación)")
print("=" * 70)

fake = Faker()
Faker.seed(42)
import random

def generate_iot_event(vehicle_id: int) -> dict:
    base_time = datetime.now() - timedelta(days=random.randint(0, 30))
    event = {
        "event_id": f"IOT-{vehicle_id}-{random.randint(10000, 99999)}",
        "vehicle_id": vehicle_id,
        "timestamp": base_time.isoformat(),
        "location": {"latitude": round(random.uniform(4.0, 8.0), 6), "longitude": round(random.uniform(-77.0, -72.0), 6), "city": random.choice(['Bogotá', 'Medellín', 'Cali', 'Barranquilla', 'Cartagena'])},
        "sensor_data": {"speed_kmh": round(random.uniform(0, 120), 2), "rpm_engine": random.randint(800, 4000), "fuel_level_percent": round(random.uniform(10, 100), 1), "engine_temp_celsius": round(random.uniform(85, 110), 1), "battery_voltage": round(random.uniform(12.0, 14.5), 2)},
        "driver_behavior": {"hard_braking": random.choice([True, False]), "hard_acceleration": random.choice([True, False]), "speeding_violation": random.choice([True, False])},
        "status": random.choice(['normal', 'warning', 'critical'])
    }
    return event

iot_events = []
for vehicle_id in range(1, 201):
    num_events = random.randint(20, 50)
    for _ in range(num_events):
        iot_events.append(generate_iot_event(vehicle_id))

print(f"\n✅ {len(iot_events):,} eventos de telemetría generados")
print("\n📋 Ejemplo de evento IoT (formato MongoDB):")
print(json.dumps(iot_events[0], indent=2))

<div class="section">
<h2>📊 9. Visualización de Métricas del Data Warehouse</h2>
</div>

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

vehicle_counts = dim_vehicle['vehicle_type'].value_counts()
axes[0, 0].bar(vehicle_counts.index, vehicle_counts.values, color=sns.color_palette("husl", len(vehicle_counts)))
axes[0, 0].set_title('Distribución de Vehículos por Tipo', fontsize=12, fontweight='bold')
axes[0, 0].tick_params(axis='x', rotation=45)

trip_status = fact_trips['status'].value_counts()
axes[0, 1].pie(trip_status.values, labels=trip_status.index, autopct='%1.1f%%', colors=sns.color_palette("Set2", len(trip_status)))
axes[0, 1].set_title('Distribución de Viajes por Estado', fontsize=12, fontweight='bold')

iot_status = pd.DataFrame(iot_events)['status'].value_counts()
axes[1, 0].bar(iot_status.index, iot_status.values, color=sns.color_palette("viridis", len(iot_status)))
axes[1, 0].set_title('Estado de Telemetría IoT', fontsize=12, fontweight='bold')

dim_type_counts = dim_route.groupby('destination_city').size().sort_values(ascending=False).head(10)
axes[1, 1].barh(dim_type_counts.index, dim_type_counts.values, color=sns.color_palette("Set3", len(dim_type_counts)))
axes[1, 1].set_title('Top 10 Ciudades Destino', fontsize=12, fontweight='bold')

plt.suptitle('FleetLogix - Métricas del Data Warehouse', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

<div class="section">
<h2>📋 10. Resumen del Pipeline ETL</h2>
</div>

In [ ]:
print("=" * 70)
print("📋 RESUMEN DEL PIPELINE ETL")
print("=" * 70)
print("""
╔══════════════════════════════════════════════════════════════════════╗
║                    PIPELINE ETL - FLEETLOGIX DWH                      ║
╠══════════════════════════════════════════════════════════════════════╣
║  EXTRACT (Extracción)                                                 ║
║  • Fuente: PostgreSQL (fleetlogix_db)                                  ║
║  • Volumen: ~540,000 registros                                        ║
╠══════════════════════════════════════════════════════════════════════╣
║  TRANSFORM (Transformación)                                           ║
║  • DIM_VEHICLE: 1,000 registros → 9 columnas                         ║
║  • DIM_DRIVER: 5,000 registros → 11 columnas                           ║
║  • DIM_ROUTE: 500 registros → 8 columnas                                ║
║  • FACT_TRIPS: 10,000 registros → 14 columnas                          ║
║  • FACT_DELIVERIES: 10,000 registros → 12 columnas                     ║
╠══════════════════════════════════════════════════════════════════════╣
║  LOAD (Carga)                                                         ║
║  • Destino: Snowflake (FLEETLOGIX_DW.ANALYTICS)                      ║
║  • Modelo: Star Schema con claves surrogate                           ║
╠══════════════════════════════════════════════════════════════════════╣
║  IoT TELEMETRY (MongoDB)                                              ║
║  • Eventos generados: 6,500+                                           ║
╚══════════════════════════════════════════════════════════════════════╝
""")
print("=" * 70)

<div class="section">
<h2>🔚 11. Cierre de Conexiones</h2>
</div>

In [ ]:
pg_conn.close()
print("✅ Conexión a PostgreSQL cerrada")

print("\n" + "=" * 70)
print("📋 RESUMEN EJECUTIVO - AVANCE 3")
print("=" * 70)
print("""
✅ DATA WAREHOUSE Y ETL COMPLETADOS

CUMPLIMIENTO DE REQUISITOS HENRY:
  ✅ Star Schema diseñado con 4 dimensiones y 2 hechos
  ✅ Esquema SQL para Snowflake generado
  ✅ Pipeline ETL (Extract, Transform, Load) implementado
  ✅ Controles de calidad de datos ejecutados
  ✅ 6,500+ eventos IoT de telemetría generados
""")
print("=" * 70)

<div class="section">
<hr>
<h3>Información del Notebook</h3>
<ul>
<li><strong>Autor:</strong> Dody Dueñas</li>
<li><strong>Proyecto:</strong> FleetLogix Enterprise - Henry M2</li>
<li><strong>Fecha:</strong> Abril 2026</li>
<li><strong>Versión:</strong> 1.0</li>
</ul>
</div>

<!-- PROFESSIONAL_FOOTER_v1 -->
---

## 🏁 Conclusiones de este Avance

- ✔ Star schema con 2 fact tables y 5 dimensiones conformadas
- ✔ Carga incremental con marcas de tiempo (watermark)
- ✔ Política SCD2 para `dim_driver` (cambios de licencia)
- ✔ Latencia E2E ingesta → DW < 15 minutos

## ➡ Siguiente paso

Continúa con **`Avance_4_CloudArchitecture.ipynb`** para profundizar en la siguiente etapa del proyecto.

---

<div align="center">

*FleetLogix · Proyecto 2 · Henry Data Science · 2026*  
📧 dodydurema67@gmail.com

</div>
